## Etapa 1: Preparação e Importação de Dados

1-


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")
sns.set_palette(sns.color_palette("deep"))

os.makedirs("graficos", exist_ok=True)   # pasta de saída (item 20)

def salvar(fig, nome):
    """Salva a figura em graficos/ com boa resolução."""
    fig.savefig(f"graficos/{nome}", dpi=150, bbox_inches="tight")

def br(v, casas=0):
    """Formata número no padrão brasileiro (1.234,56)."""
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

MESES = ["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
         "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]

2-


In [ ]:

rng = np.random.default_rng(42)

lojas = {  # loja: (região, peso de vendas)
    "Loja São Paulo":      ("Sudeste",      1.6),
    "Loja Belo Horizonte": ("Sudeste",      1.2),
    "Loja Curitiba":       ("Sul",          1.0),
    "Loja Recife":         ("Nordeste",     0.9),
    "Loja Brasília":       ("Centro-Oeste", 0.8),
    "Loja Manaus":         ("Norte",        0.5),
}
produtos = {  
    "Notebook":              ("Eletrônicos", 3500, 1, 4),
    "Smartphone":            ("Eletrônicos", 2200, 1, 5),
    "Fone de Ouvido":        ("Eletrônicos",  250, 1, 8),
    "Cadeira de Escritório": ("Móveis",       850, 1, 4),
    "Mesa de Escritório":    ("Móveis",      1200, 1, 3),
    "Caderno":               ("Papelaria",     25, 2, 30),
    "Caneta":                ("Papelaria",      5, 5, 60),
    "Mochila":               ("Acessórios",   180, 1, 10),
    "Mouse":                 ("Acessórios",    90, 1, 12),
}
n = 800
peso_mes = np.array([0.8, 0.8, 0.9, 0.9, 1.0, 0.9, 0.9, 0.9, 1.0, 1.1, 1.5, 1.8])  # mais vendas em nov/dez
meses = rng.choice(np.arange(1, 13), size=n, p=peso_mes / peso_mes.sum())
dias = rng.integers(1, 29, size=n)
nomes_lojas = list(lojas)
p_lojas = np.array([lojas[l][1] for l in nomes_lojas]); p_lojas = p_lojas / p_lojas.sum()
loja = rng.choice(nomes_lojas, size=n, p=p_lojas)
produto = rng.choice(list(produtos), size=n)

dados = pd.DataFrame({
    "data": pd.to_datetime({"year": 2025, "month": meses, "day": dias}).dt.strftime("%Y-%m-%d"),
    "loja": loja,
    "regiao": [lojas[l][0] for l in loja],
    "produto": produto,
    "categoria": [produtos[p][0] for p in produto],
    "quantidade": [int(rng.integers(produtos[p][2], produtos[p][3] + 1)) for p in produto],
    "preco_unitario": [round(produtos[p][1] * rng.normal(1, 0.08), 2) for p in produto],
})

dados.loc[rng.choice(n, 15, replace=False), "preco_unitario"] = np.nan
dados.loc[rng.choice(n, 15, replace=False), "quantidade"] = np.nan
dados.to_csv("vendas.csv", index=False)


df = pd.read_csv("vendas.csv")
df["data"] = pd.to_datetime(df["data"])
df.head()

3-


In [ ]:
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
df[["quantidade", "preco_unitario", "faturamento"]].head()

4-


In [ ]:
print("=== .info() ===")
df.info()

In [ ]:
print("=== Valores ausentes por coluna ===")
print(df.isnull().sum())

In [ ]:
print("=== Resumo estatístico ===")
df.describe()

5-


In [ ]:
df["preco_unitario"] = df["preco_unitario"].fillna(df["preco_unitario"].mean())
df["quantidade"] = df["quantidade"].fillna(df["quantidade"].mean())
df["faturamento"] = df["quantidade"] * df["preco_unitario"]   

print("Nulos restantes:", int(df.isnull().sum().sum()))

## Etapa 2: Análise Exploratória e Estatísticas

6-


In [ ]:
fat_loja = df.groupby("loja")["faturamento"].sum().sort_values(ascending=False)
fat_loja.round(2)

7-


In [ ]:
ticket_medio = (df.groupby("loja")["faturamento"].sum() / df.groupby("loja").size()).sort_values(ascending=False)
ticket_medio.round(2)

8-


In [ ]:
top5_produtos = df.groupby("produto")["quantidade"].sum().sort_values(ascending=False).head(5)
top5_produtos.round(0)

9-


In [ ]:
fat_ano_mes = df.groupby(df["data"].dt.to_period("M"))["faturamento"].sum()

def mes_ano(p):
    return f"{MESES[p.month - 1]}/{p.year}"

mes_maior = fat_ano_mes.idxmax()
mes_menor = fat_ano_mes.idxmin()
print(f"Maior faturamento: {mes_ano(mes_maior)} (R$ {br(fat_ano_mes.max(), 2)})")
print(f"Menor faturamento: {mes_ano(mes_menor)} (R$ {br(fat_ano_mes.min(), 2)})")

10-


In [ ]:
df["mes"] = df["data"].dt.month
df[["data", "mes"]].head()

## Etapa 3: Visualizações com matplotlib e seaborn

11-


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=fat_loja.index, y=fat_loja.values, hue=fat_loja.index, palette="deep", legend=False, ax=ax)
for i, v in enumerate(fat_loja.values):
    ax.text(i, v, f"R$ {br(v / 1000)} mil", ha="center", va="bottom", fontsize=9)
ax.set_title("Faturamento total por loja em 2025")
ax.set_xlabel("Loja")
ax.set_ylabel("Faturamento (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
salvar(fig, "faturamento_por_loja.png")
plt.show()

12-


In [ ]:
fat_mensal = df.groupby("mes")["faturamento"].sum()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(fat_mensal.index, fat_mensal.values, marker="o", linewidth=2, color=sns.color_palette("deep")[0])
for x, y in zip(fat_mensal.index, fat_mensal.values):
    ax.annotate(f"{br(y / 1000)} mil", (x, y), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels([m[:3] for m in MESES])
ax.set_title("Evolução mensal do faturamento em 2025")
ax.set_xlabel("Mês")
ax.set_ylabel("Faturamento (R$)")
ax.set_ylim(0, fat_mensal.max() * 1.15)
plt.tight_layout()
salvar(fig, "evolucao_mensal_faturamento.png")
plt.show()

13-


In [ ]:
fat_regiao = df.groupby("regiao")["faturamento"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(fat_regiao.values, labels=fat_regiao.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("deep", len(fat_regiao)), wedgeprops={"edgecolor": "white"})
ax.set_title("Distribuição do faturamento por região")
plt.tight_layout()
salvar(fig, "faturamento_por_regiao_pizza.png")
plt.show()

14-


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x="quantidade", y="faturamento", hue="categoria", alpha=0.7, ax=ax)
ax.set_title("Relação entre quantidade vendida e faturamento por venda")
ax.set_xlabel("Quantidade de itens vendidos")
ax.set_ylabel("Faturamento da venda (R$)")
ax.legend(title="Categoria")
plt.tight_layout()
salvar(fig, "dispersao_quantidade_faturamento.png")
plt.show()

15-


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x="categoria", y="preco_unitario", hue="categoria", palette="deep", legend=True, ax=ax)
ax.set_yscale("log")   # escala log: os preços variam de poucos reais a milhares
ax.set_title("Distribuição dos preços unitários por categoria de produto")
ax.set_xlabel("Categoria")
ax.set_ylabel("Preço unitário (R$, escala logarítmica)")
ax.legend(title="Categoria", loc="upper right")
plt.tight_layout()
salvar(fig, "boxplot_preco_por_categoria.png")
plt.show()

16-


In [ ]:
corr = df[["quantidade", "preco_unitario", "faturamento", "mes"]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlação entre variáveis numéricas")
plt.tight_layout()
salvar(fig, "heatmap_correlacao.png")
plt.show()

## Etapa 4: Estilização e Apresentação

17-


In [ ]:
sns.set_style("whitegrid")
sns.set_palette(sns.color_palette("deep"))
print("Tema global: whitegrid | Paleta: deep")

18-
19-


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))


sns.barplot(x=fat_loja.index, y=fat_loja.values, hue=fat_loja.index, palette="deep", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Faturamento total por loja")
axes[0, 0].set_xlabel("Loja")
axes[0, 0].set_ylabel("Faturamento (R$)")
axes[0, 0].tick_params(axis="x", rotation=25)


sns.barplot(x=fat_regiao.index, y=fat_regiao.values, hue=fat_regiao.index, palette="muted", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Faturamento total por região")
axes[0, 1].set_xlabel("Região")
axes[0, 1].set_ylabel("Faturamento (R$)")


axes[1, 0].plot(fat_mensal.index, fat_mensal.values, marker="o", linewidth=2)
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels([m[:3] for m in MESES])
axes[1, 0].set_title("Evolução mensal do faturamento")
axes[1, 0].set_xlabel("Mês")
axes[1, 0].set_ylabel("Faturamento (R$)")


sns.barplot(x=top5_produtos.values, y=top5_produtos.index, hue=top5_produtos.index, palette="deep", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Top 5 produtos mais vendidos (quantidade)")
axes[1, 1].set_xlabel("Unidades vendidas")
axes[1, 1].set_ylabel("Produto")

fig.suptitle("Painel de vendas 2025: visão geral", fontsize=16, y=1.0)
plt.tight_layout()
salvar(fig, "painel_geral.png")
plt.show()

20-


In [ ]:
linhas = []
linhas.append({"indicador": "Faturamento total", "detalhe": "todas as lojas", "valor": round(df["faturamento"].sum(), 2)})
linhas.append({"indicador": "Número de vendas", "detalhe": "todas as lojas", "valor": len(df)})
for loja_, v in fat_loja.items():
    linhas.append({"indicador": "Faturamento por loja", "detalhe": loja_, "valor": round(v, 2)})
for loja_, v in ticket_medio.items():
    linhas.append({"indicador": "Ticket médio por loja", "detalhe": loja_, "valor": round(v, 2)})
for prod, v in top5_produtos.items():
    linhas.append({"indicador": "Top 5 produtos (quantidade)", "detalhe": prod, "valor": round(v, 0)})
for reg, v in fat_regiao.items():
    linhas.append({"indicador": "Faturamento por região", "detalhe": reg, "valor": round(v, 2)})
linhas.append({"indicador": "Mês de maior faturamento", "detalhe": mes_ano(mes_maior), "valor": round(fat_ano_mes.max(), 2)})
linhas.append({"indicador": "Mês de menor faturamento", "detalhe": mes_ano(mes_menor), "valor": round(fat_ano_mes.min(), 2)})

resumo = pd.DataFrame(linhas)
resumo.to_csv("resumo_analitico.csv", index=False, encoding="utf-8-sig")
print("Arquivos em graficos/:", sorted(os.listdir("graficos")))
resumo